# 01 - PCOS Data Audit and EDA

Purpose:

- Load the required PCOS clinical dataset.
- Clean the raw workbook into an analysis-ready table.
- Audit missingness, class balance, outliers, and silently coerced values.
- Save a processed CSV for later notebooks.

Run this notebook first.


In [ ]:
from pathlib import Path
import re
import json
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "_read_extract"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
METRIC_DIR = OUTPUT_DIR / "metrics"

for folder in [RAW_DIR, OUTPUT_DIR, FIGURE_DIR, MODEL_DIR, METRIC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ZIP_PATH = PROJECT_ROOT / "OneDrive_1_5-11-2026.zip"
PCOS_XLSX = RAW_DIR / "(Main_Dataset)_PCOS_data_without_infertility.xlsx"
ENDO_CSV = RAW_DIR / "(Supplementary_Dataset)_structured_endometriosis_data.csv"

def ensure_small_datasets_extracted():
    """Extract only the tabular datasets if they are not already available."""
    if PCOS_XLSX.exists() and ENDO_CSV.exists():
        return
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"Cannot find {ZIP_PATH}")
    targets = {
        "(Main_Dataset)_PCOS_data_without_infertility.xlsx": PCOS_XLSX,
        "(Supplementary_Dataset)_structured_endometriosis_data.csv": ENDO_CSV,
    }
    with zipfile.ZipFile(ZIP_PATH) as zf:
        for member, destination in targets.items():
            if not destination.exists():
                with zf.open(member) as src, open(destination, "wb") as dst:
                    dst.write(src.read())

ensure_small_datasets_extracted()

def clean_column_name(name):
    name = str(name).strip().lower()
    name = name.replace("β", "beta")
    name = name.replace("marraige", "marriage")
    name = name.replace("bp _", "bp ")
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

# Columns we never want to keep, even though they parse as numeric.
# - sl_no / patient_file_no: identifiers, no clinical signal.
# - blood_group: stored as ordinal codes (11-18) which has no meaningful order.
# - marriage_status_yrs: clinically sensitive and a poor proxy for anything
#   diagnostic; the plan flags it as unsuitable for screening.
DROP_COLUMNS = ["sl_no", "patient_file_no", "blood_group", "marriage_status_yrs"]

def load_pcos_raw():
    df = pd.read_excel(PCOS_XLSX, sheet_name="Full_new", engine="openpyxl")
    df = df.dropna(how="all").copy()
    df.columns = [clean_column_name(c) for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def clean_pcos_dataframe(df):
    df = df.copy()

    # Track silently-coerced non-numeric cells so the audit can flag them.
    coercion_report = {}
    for col in df.columns:
        before_non_null = df[col].notna().sum()
        coerced = pd.to_numeric(df[col], errors="coerce")
        new_nan = coerced.isna().sum() - df[col].isna().sum()
        if new_nan > 0:
            offending = df.loc[df[col].notna() & coerced.isna(), col].astype(str).unique().tolist()
            coercion_report[col] = {"coerced_to_nan": int(new_nan), "examples": offending[:5]}
        df[col] = coerced

    drop_now = [c for c in DROP_COLUMNS if c in df.columns]
    df = df.drop(columns=drop_now)

    if "pcos_y_n" not in df.columns:
        raise ValueError("Expected target column 'pcos_y_n' after cleaning column names.")

    df = df[df["pcos_y_n"].isin([0, 1])].copy()
    df["pcos_y_n"] = df["pcos_y_n"].astype(int)

    # Kaggle-style encoding uses 2 for regular and 4/5 for irregular cycles.
    if "cycle_r_i" in df.columns:
        df["cycle_irregular_flag"] = np.where(
            df["cycle_r_i"].isna(),
            np.nan,
            np.where(df["cycle_r_i"] >= 4, 1, 0),
        )

    # Conservative caps for visibly impossible or model-dominating outliers.
    caps = {
        "fsh_miu_ml": (0, 100),
        "lh_miu_ml": (0, 200),
        "fsh_lh": (0, 50),
        "vit_d3_ng_ml": (0, 150),
        "prg_ng_ml": (0, 50),
    }
    for col, (low, high) in caps.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=low, upper=high)

    df.attrs["coercion_report"] = coercion_report
    df.attrs["dropped_columns"] = drop_now
    return df

pcos_raw = load_pcos_raw()
pcos = clean_pcos_dataframe(pcos_raw)
print("PCOS shape:", pcos.shape)
print("Dropped columns:", pcos.attrs["dropped_columns"])
if pcos.attrs["coercion_report"]:
    print("Non-numeric values coerced to NaN:")
    for col, info in pcos.attrs["coercion_report"].items():
        print(f"  {col}: {info['coerced_to_nan']} cell(s); examples={info['examples']}")
print(pcos["pcos_y_n"].value_counts().sort_index())


## Save Processed Dataset


In [ ]:
processed_path = OUTPUT_DIR / "pcos_cleaned.csv"
pcos.to_csv(processed_path, index=False)
print("Saved:", processed_path)
pcos.head()


## Column Audit


In [ ]:
audit = pd.DataFrame({
    "column": pcos.columns,
    "dtype": [str(pcos[c].dtype) for c in pcos.columns],
    "missing": [pcos[c].isna().sum() for c in pcos.columns],
    "missing_pct": [pcos[c].isna().mean() for c in pcos.columns],
    "n_unique": [pcos[c].nunique(dropna=True) for c in pcos.columns],
})
audit.to_csv(METRIC_DIR / "pcos_column_audit.csv", index=False)
audit


## Data Quality Issues Coerced During Cleaning

Non-numeric strings in numeric columns silently become NaN. These are logged for the slide-deck data-quality discussion.


In [ ]:
coercion_report = pcos.attrs.get("coercion_report", {})
if coercion_report:
    coercion_df = pd.DataFrame([
        {"column": col, "coerced_to_nan": info["coerced_to_nan"], "examples": ", ".join(info["examples"])}
        for col, info in coercion_report.items()
    ])
    coercion_df.to_csv(METRIC_DIR / "pcos_coercion_report.csv", index=False)
    display(coercion_df)
else:
    print("No non-numeric values were coerced.")
print("Columns intentionally dropped:", pcos.attrs.get("dropped_columns", []))


## Class Balance


In [ ]:
class_counts = pcos["pcos_y_n"].value_counts().sort_index()
display(class_counts.rename({0: "non_pcos", 1: "pcos"}))

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind="bar", ax=ax, color=["#3a7ca5", "#d95f59"])
ax.set_title("PCOS target distribution")
ax.set_xlabel("PCOS status")
ax.set_ylabel("Patients")
ax.set_xticklabels(["No PCOS", "PCOS"], rotation=0)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pcos_class_balance.png", dpi=160)
plt.show()


## Feature Groups


In [ ]:
TARGET = "pcos_y_n"

# fast_food_y_n and reg_exercise_y_n are deliberately excluded.
# The plan flags lifestyle proxies as bias-prone and likely to stigmatise patients.
SCREENING_FEATURES = [
    "age_yrs",
    "bmi",
    "cycle_r_i",
    "cycle_irregular_flag",
    "cycle_length_days",
    "weight_gain_y_n",
    "hair_growth_y_n",
    "skin_darkening_y_n",
    "hair_loss_y_n",
    "pimples_y_n",
    "rbs_mg_dl",
    "bp_systolic_mmhg",
    "bp_diastolic_mmhg",
]

ENHANCED_EXTRA_FEATURES = [
    "hb_g_dl",
    "fsh_miu_ml",
    "lh_miu_ml",
    "fsh_lh",
    "tsh_miu_l",
    "amh_ng_ml",
    "prl_ng_ml",
    "vit_d3_ng_ml",
    "prg_ng_ml",
    "follicle_no_l",
    "follicle_no_r",
    "avg_f_size_l_mm",
    "avg_f_size_r_mm",
    "endometrium_mm",
]

def available(features, df):
    return [feature for feature in features if feature in df.columns]

screening_features = available(SCREENING_FEATURES, pcos)
enhanced_features = available(SCREENING_FEATURES + ENHANCED_EXTRA_FEATURES, pcos)

print("Screening features:", screening_features)
print("Enhanced features:", enhanced_features)


## Summary Statistics by PCOS Status


In [ ]:
summary_features = [
    "age_yrs", "bmi", "cycle_r_i", "cycle_irregular_flag", "cycle_length_days",
    "fsh_miu_ml", "lh_miu_ml", "fsh_lh", "amh_ng_ml", "rbs_mg_dl",
    "follicle_no_l", "follicle_no_r", "endometrium_mm",
]
summary_features = available(summary_features, pcos)
summary = pcos.groupby("pcos_y_n")[summary_features].agg(["mean", "median", "std"]).T
summary.to_csv(METRIC_DIR / "pcos_grouped_summary.csv")
summary


## Symptom Prevalence by PCOS Status


In [ ]:
symptom_cols = available([
    "weight_gain_y_n", "hair_growth_y_n", "skin_darkening_y_n",
    "hair_loss_y_n", "pimples_y_n", "reg_exercise_y_n",
], pcos)

symptom_rates = pcos.groupby("pcos_y_n")[symptom_cols].mean().T
symptom_rates.columns = ["non_pcos_rate", "pcos_rate"]
symptom_rates["absolute_difference"] = symptom_rates["pcos_rate"] - symptom_rates["non_pcos_rate"]
symptom_rates = symptom_rates.sort_values("absolute_difference", ascending=False)
symptom_rates.to_csv(METRIC_DIR / "pcos_symptom_rates.csv")
display(symptom_rates)

fig, ax = plt.subplots(figsize=(8, 5))
symptom_rates[["non_pcos_rate", "pcos_rate"]].plot(kind="barh", ax=ax)
ax.set_title("Symptom rates by PCOS status")
ax.set_xlabel("Rate")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pcos_symptom_rates.png", dpi=160)
plt.show()


## Outlier Review


In [ ]:
outlier_cols = available([
    "fsh_miu_ml", "lh_miu_ml", "fsh_lh", "vit_d3_ng_ml", "prg_ng_ml",
    "i_beta_hcg_miu_ml", "ii_beta_hcg_miu_ml",
], pcos)

outlier_table = pcos[outlier_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
outlier_table.to_csv(METRIC_DIR / "pcos_outlier_review.csv")
outlier_table


## Correlation Snapshot


In [ ]:
numeric = pcos.select_dtypes(include=[np.number])
top_corr = numeric.corr(numeric_only=True)["pcos_y_n"].drop("pcos_y_n").abs().sort_values(ascending=False).head(20)
display(top_corr)

plot_cols = ["pcos_y_n"] + top_corr.index.tolist()[:12]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(numeric[plot_cols].corr(), cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation heatmap for top PCOS-associated features")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pcos_correlation_heatmap.png", dpi=160)
plt.show()
